# Dangerous Sound Detection MVP - Exploratory Notebook

This notebook explores the dangerous sound detection project, demonstrating data loading, preprocessing, feature extraction, training, and inference.

## 0. Notebook Notes

Run the notebook from top to bottom. The first code cell sets up imports, paths, and shared variables used by the rest of the notebook.


## 1. Project Setup


In [ ]:
import sys
from pathlib import Path

import json
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError('Run this notebook from the project root or the notebooks directory.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import load_config
from src.features.yamnet_extractor import YAMNetExtractor
from src.inference.predict_audio import predict_audio
from src.inference.predict_video import predict_video
from src.utils.audio import load_audio, preprocess_audio, window_audio

config = load_config(str(PROJECT_ROOT / 'configs' / 'default.yaml'))
raw_dir = PROJECT_ROOT / 'data' / 'raw'
processed_dir = PROJECT_ROOT / 'data' / 'processed'
models_dir = PROJECT_ROOT / 'models'
outputs_dir = PROJECT_ROOT / 'outputs'

sample_file = None
video_file = None
extractor = None
features = None
labels = None
le = None
X_train = None
X_test = None
y_train = None
y_test = None
clf = None

print(f'Project root: {PROJECT_ROOT}')
print(f'Classes: {config["class_names"]}')


## 2. Load and Explore Dataset

In [ ]:
# Inspect class balance
class_counts = {}
for class_name in config['class_names']:
    class_dir = raw_dir / class_name
    class_counts[class_name] = len(list(class_dir.glob('*'))) if class_dir.exists() else 0

print('Class balance:')
for cls, count in class_counts.items():
    print(f'{cls}: {count} files')

plt.figure(figsize=(8, 4))
plt.bar(class_counts.keys(), class_counts.values())
plt.title('Class Balance')
plt.xlabel('Class')
plt.ylabel('Number of Files')
plt.show()

sample_file = None
for class_name in config['class_names']:
    class_dir = raw_dir / class_name
    if class_dir.exists():
        files = sorted(class_dir.glob('*.wav'))
        if files:
            sample_file = files[0]
            break

if sample_file is not None:
    y = load_audio(str(sample_file), sr=config['sample_rate'])
    sr = config['sample_rate']
    print(f'Sample file: {sample_file}')
    print(f'Duration: {len(y) / sr:.2f} seconds')
    print(f'Sample rate: {sr} Hz')

    plt.figure(figsize=(10, 4))
    plt.plot(np.arange(len(y)) / sr, y)
    plt.title('Waveform')
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude')
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.specgram(y, Fs=sr, NFFT=1024, noverlap=512, cmap='viridis')
    plt.title('Spectrogram')
    plt.xlabel('Time (s)')
    plt.ylabel('Frequency (Hz)')
    plt.colorbar(label='Intensity (dB)')
    plt.show()
else:
    print('No sample audio file found.')


## 3. Audio Preprocessing

In [ ]:
# Demonstrate preprocessing on sample
if sample_file is not None:
    y_orig = load_audio(str(sample_file), sr=config['sample_rate'])
    print(f'Original: shape={y_orig.shape}, sr={config["sample_rate"]}')

    y_proc = preprocess_audio(y_orig, config['sample_rate'])
    print(f'Preprocessed: shape={y_proc.shape}, max_abs={np.max(np.abs(y_proc)):.3f}')

    frames = window_audio(y_proc, config['window_length'], config['hop_length'], config['sample_rate'])
    print(f'Frames: shape={frames.shape}, n_frames={frames.shape[0]}')

    plt.figure(figsize=(10, 3))
    plt.plot(frames[0])
    plt.title('First Windowed Frame')
    plt.xlabel('Samples')
    plt.ylabel('Amplitude')
    plt.show()
else:
    print('No sample for preprocessing demo.')


## 4. Feature Extraction with YAMNet

In [ ]:
# Load YAMNet and extract embeddings for a sample
if sample_file is not None:
    try:
        extractor = YAMNetExtractor(config['yamnet_model_url'])
        y_proc = preprocess_audio(load_audio(str(sample_file), sr=config['sample_rate']), config['sample_rate'])
        frames = window_audio(y_proc, config['window_length'], config['hop_length'], config['sample_rate'])

        embeddings = []
        for frame in frames[:5]:
            emb = extractor.extract(frame, config['sample_rate'])
            embeddings.append(emb.mean(axis=0))

        embeddings = np.asarray(embeddings)
        print(f'Embeddings shape: {embeddings.shape}')

        plt.figure(figsize=(10, 4))
        plt.imshow(embeddings.T, aspect='auto', cmap='viridis')
        plt.title('YAMNet Embeddings (First 5 Frames)')
        plt.xlabel('Frame')
        plt.ylabel('Embedding Dimension')
        plt.colorbar()
        plt.show()
    except Exception as exc:
        print(f'Could not load YAMNet demo: {exc}')
else:
    print('No sample for feature extraction demo.')


## 5. Prepare Training Data

In [ ]:
# Load processed features and labels produced by the CLI pipeline
features_path = processed_dir / 'features.npy'
labels_path = processed_dir / 'labels.npy'

if features_path.exists() and labels_path.exists():
    features = np.load(features_path)
    labels = np.load(labels_path, allow_pickle=True)
    print(f'Features shape: {features.shape}')
    print(f'Labels shape: {labels.shape}')

    le = LabelEncoder()
    labels_encoded = le.fit_transform(labels)
    print(f'Classes: {le.classes_}')

    X_train, X_test, y_train, y_test = train_test_split(
        features,
        labels_encoded,
        test_size=0.2,
        random_state=42,
        stratify=labels_encoded,
    )
    print(f'Train: {X_train.shape}, Test: {X_test.shape}')
else:
    print('Processed features not found. Run the feature extraction CLI step first.')


## 6. Train Classifier

In [ ]:
# Train LogisticRegression
if X_train is not None and y_train is not None and le is not None:
    clf = LogisticRegression(random_state=42, max_iter=1000)
    clf.fit(X_train, y_train)
    print('Model trained.')

    models_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(clf, models_dir / 'demo_classifier.pkl')
    joblib.dump(le, models_dir / 'demo_label_encoder.pkl')
    print('Demo model saved.')
else:
    print('Training data not available.')


## 7. Evaluate Model

In [ ]:
# Evaluate
if clf is not None and X_test is not None and y_test is not None and le is not None:
    preds = clf.predict(X_test)
    acc = accuracy_score(y_test, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, preds, average='macro', zero_division=0)
    print(f'Accuracy: {acc:.3f}')
    print(f'Precision: {prec:.3f}')
    print(f'Recall: {rec:.3f}')
    print(f'F1: {f1:.3f}')

    cm = confusion_matrix(y_test, preds)
    plt.figure(figsize=(6, 5))
    plt.imshow(cm, interpolation='nearest')
    plt.title('Confusion Matrix')
    plt.colorbar()
    plt.xticks(range(len(le.classes_)), le.classes_)
    plt.yticks(range(len(le.classes_)), le.classes_)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()

    print(classification_report(y_test, preds, target_names=le.classes_, zero_division=0))
else:
    print('Model not trained.')


## 8. Inference on Audio File

In [ ]:
# Demo inference on one audio file
if sample_file is not None:
    try:
        result = predict_audio(str(sample_file), config)
        print('Inference result:')
        print(result)
    except FileNotFoundError:
        print('Model not found. Run training first.')
else:
    print('No sample file for inference.')


## 9. Inference on Video File

In [ ]:
# Find a video file
video_file = None
video_patterns = ('*.mp4', '*.avi', '*.mov', '*.mkv')
for class_name in config['class_names']:
    class_dir = raw_dir / class_name
    if not class_dir.exists():
        continue
    for pattern in video_patterns:
        files = sorted(class_dir.glob(pattern))
        if files:
            video_file = files[0]
            break
    if video_file is not None:
        break

if video_file is not None:
    try:
        result = predict_video(str(video_file), config)
        print('Video inference result:')
        print(result)
    except FileNotFoundError:
        print('Model not found.')
else:
    print('No video file found.')


## 10. Inference on Folder of Files

In [ ]:
# Batch inference on a small folder sample
outputs_dir.mkdir(parents=True, exist_ok=True)

for class_name in config['class_names']:
    class_dir = raw_dir / class_name
    if class_dir.exists():
        for file_path in list(class_dir.glob('*'))[:2]:
            try:
                if file_path.suffix.lower() in ['.wav', '.mp3']:
                    result = predict_audio(str(file_path), config)
                elif file_path.suffix.lower() in ['.mp4', '.avi', '.mov', '.mkv']:
                    result = predict_video(str(file_path), config)
                else:
                    continue

                output_file = outputs_dir / f'{file_path.stem}_result.json'
                with open(output_file, 'w', encoding='utf-8') as f:
                    json.dump(result, f, indent=2)
                print(f'Saved result to {output_file}')
            except FileNotFoundError:
                print('Model not found.')
                break
        else:
            continue
        break


## 11. Inference on RTSP Stream

In [ ]:
# RTSP inference (MVP - requires a trained model)
print('RTSP inference is implemented in src/inference/predict_stream.py')
print('Run it from the terminal:')
print('python -m src.cli predict-stream --rtsp rtsp://example.com/stream --config configs/default.yaml')
